![image_1780820700920.png](./image_1780820700920.png "image_1780820700920.png")

![image_1780820721337.png](./image_1780820721337.png "image_1780820721337.png")

![image_1780820742328.png](./image_1780820742328.png "image_1780820742328.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

# Initialize Spark session
spark = SparkSession.builder.appName("ActivitiesData").getOrCreate()

# Define dataset for activities
activities_data = [
    (1, 101, "send", 4.0),
    (2, 101, "open", 2.5),
    (3, 102, "send", 1.5),
    (4, 102, "open", 3.0),
    (5, 103, "chat", 5.0),
    (6, 103, "send", 6.0),
    (7, 103, "open", 1.0),
    (8, 104, "open", 4.0),
    (9, 104, "send", 2.0)
]

activities_columns = ["activity_id", "user_id", "activity_type", "time_spent"]

# Create activities DataFrame
activities_df = spark.createDataFrame(activities_data, activities_columns)

# Define dataset for age_breakdown
age_data = [
    (101, "21-25"),
    (102, "21-25"),
    (103, "26-30"),
    (104, "26-30")
]

age_columns = ["user_id", "age_bucket"]

# Create age_breakdown DataFrame
age_df = spark.createDataFrame(age_data, age_columns)

# Show both DataFrames
print("Activities DataFrame:")
activities_df.show()

print("Age Breakdown DataFrame:")
age_df.show()


In [0]:
result_df = (
    activities_df.join(age_df, on="user_id")
    .withColumn(
        "send_ct",
        f.sum(f.when(f.col("activity_type") == "send", f.col("time_spent"))).over(
            Window.partitionBy("age_bucket")
        ),
    )
    .withColumn(
        "open_ct",
        f.sum(f.when(f.col("activity_type") == "open", f.col("time_spent"))).over(
            Window.partitionBy("age_bucket")
        ),
    )
    .select(
        f.col("age_bucket"),
        f.round(
            f.col("send_ct") / (f.col("open_ct") + f.col("send_ct")) * 100, 2
        ).alias("send_perc"),
        f.round(
            f.col("open_ct") / (f.col("open_ct") + f.col("send_ct")) * 100, 2
        ).alias("open_perc"),
    )
    .distinct()
)
display(result_df)